# Query understanding - large, multilabel classification (hallucinate, then resolve)

1. Map a query -> N classifications, by first generating hallucinations, then resolving them to real classifications
2. Then boost matches
3. Analyze its impact to search

## Boilerplate

Install deps, mount GDrive, prompt for your OpenAI Key (placed in your GDrive)

In [ ]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)
from cheat_at_search.search import run_strategy, graded_bm25, ndcgs, ndcg_delta, vs_ideal
from cheat_at_search.wands_data import corpus, judgments

corpus

  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-7d7678xh
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-7d7678xh
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 611269f6b5f0857681d774e990312c512f0e9ca2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 13.3 MB/s eta 0:00:00
  Created wheel for cheat_at_search: filename=cheat_at_search-0.1.0-py3-none-any.whl size=1455859 sha256=73e6fb0160c77d11b4b025a554bfb1fd86c5a98cb6e5f1b27f61d54746c6fef7
  Stored in directory: /tmp/pip-ephem-wheel-cache-uwfvp5i8/wheels/ce/b7/b5/5b6456ba5de93e3b9f0b94ddbeb32ac5a07345c3

## Query -> LIST of full classification

Previously we mapped a query to a single classification. Let's now generate a LIST of possible classifications.

**Notice diff from previous**

* Our pydantic type returns a LIST, not just a single FullyClassifiedClassifications

### Get top N classifications

In [ ]:
corpus['category hierarchy'].value_counts()

## Build a vector index of classifications

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
classifications_list = corpus['category hierarchy'].value_counts().index.tolist()
classifications_embeddings = model.encode(classifications_list)

## Prompt for fake classifications

In [ ]:
from pydantic import BaseModel, Field
import numpy as np

class QueryClassification(BaseModel):
    """
    Represents a classification of a product.

    In this case, hallucinated, something the model is making up that looks like one
    of our classifications
    """
    hallucinated_classification: list[str] = Field(
        description="The classification you created for the query."
    )

In [ ]:
from openai import OpenAI
from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.enrich import AutoEnricher


def get_prompt(query):
        prompt = f"""

        Some inspiration on what these look like is at the bottom.

        Here is the users request:

        {query}

        Product classifications might look like:

        'Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables'
        'Décor & Pillows / Decorative Pillows & Blankets / Throw Pillows'
        'Furniture / Bedroom Furniture / Dressers & Chests'
        'Outdoor / Outdoor & Patio Furniture / Patio Furniture Sets / Patio Conversation Sets'
        'Home Improvement / Bathroom Remodel & Bathroom Fixtures / Bathroom Vanities / All Bathroom Vanities'
        'Lighting / Wall Lights / Bathroom Vanity Lighting'
        'Kitchen & Tabletop / Kitchen Organization / Food Storage & Canisters'
        'School Furniture and Supplies / School Furniture / School Chairs & Seating / Stackable Chairs',
        'Baby & Kids / Toddler & Kids Bedroom Furniture / Kids Beds',

        If you feel inspired, return many unique values in a list. Be creative. Cast a wide net with related but diverse categories.

        Return empty list if no clear classification could be inferred, the query is not clearly a furniture query

        """

        return prompt


def resolve(hallucinated: list[str], threshold=0.92):
    actuals = []
    for halluc_class in hallucinated:
        query_embedding = model.encode(halluc_class)
        dot_prods = np.dot(classifications_embeddings, query_embedding)
        actual = classifications_list[np.argmax(dot_prods)]
        while actual in actuals:
            dot_prods[np.argmax(dot_prods)] = -1
            actual = classifications_list[np.argmax(dot_prods)]
        dot_prod = np.max(dot_prods)
        if dot_prod > threshold:
            actuals.append(actual)
    return actuals

system_prompt="""Your task is to create novel, never seen before, furniture,
                home goods, or hardware classification that best fit a search query. """


enricher = AutoEnricher(
    model="openai/gpt-5.4-nano",
    system_prompt=system_prompt,
    response_model=QueryClassification,
    reasoning_effort="none"
)

def classify(query):
    enriched = enricher.enrich(get_prompt(query))

    classifications = resolve(enriched.hallucinated_classification)
    return classifications

classified = classify("desk for kids")
classified

## Run Category search strategy with classifier

Let's search, providing a boost when the product matches the category

In [ ]:
from searcharray import SearchArray
from cheat_at_search.wands_data import corpus, judgments
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.strategy.strategy import SearchStrategy
import numpy as np


def parse_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return toked[0]
    return None

def parse_sub_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return '/'.join(toked[1:])
    return None

class CategorySearch(SearchStrategy):
    def __init__(self, products, query_to_cat,
                 name_boost=9.3,
                 description_boost=4.1,
                 category_boost=0,
                 sub_category_boost=0,
                 direct_boost=10):
        super().__init__(products, workers=16)
        self.index = products
        if 'title_snowball' not in self.index.columns:
            self.index['title_snowball'] = SearchArray.index(
                products['title'], snowball_tokenizer)
        if 'description_snowball' not in self.index.columns:
            self.index['description_snowball'] = SearchArray.index(
                products['description'], snowball_tokenizer)

        cat_split = products['category hierarchy'].fillna('').str.split("/")
        self.index['category_snowball'] = SearchArray.index(
            products['category'], snowball_tokenizer
        )
        self.index['sub_category_snowball'] = SearchArray.index(
            products['sub_category'], snowball_tokenizer
        )
        self.index['classification_snowball'] = SearchArray.index(
            products['category hierarchy'], snowball_tokenizer
        )

        self.query_to_cat = query_to_cat
        self.name_boost = name_boost
        self.description_boost = description_boost
        self.category_boost = category_boost
        self.sub_category_boost = sub_category_boost
        self.direct_boost = direct_boost

    def search(self, query, k=10):
        """Dumb baseline lexical search, but add a constant boost when
           the desired category or subcategory"""
        bm25_scores = np.zeros(len(self.index))
        classifications = self.query_to_cat(query)
        categories = [parse_category(classification) for classification in classifications]
        sub_categories = [parse_sub_category(classification) for classification in classifications]

        categories = set(categories)
        sub_categories = set(sub_categories)

        tokenized = snowball_tokenizer(query)

        # ****
        # Baseline BM25 search from before
        for token in tokenized:
            bm25_scores += self.index['title_snowball'].array.score(token) * self.name_boost
            bm25_scores += self.index['description_snowball'].array.score(
                token) * self.description_boost

        # ****
        # If there's a category, boost that by a constant amount
        for category in categories:
            tokenized_category = snowball_tokenizer(category)
            category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_category:
                category_match = self.index['category_snowball'].array.score(tokenized_category) > 0
            bm25_scores[category_match] += self.category_boost

        for sub_category in sub_categories:
            tokenized_sub_category = snowball_tokenizer(sub_category)
            sub_category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_sub_category:
                sub_category_match = self.index['category_snowball'].array.score(tokenized_sub_category) > 0
            bm25_scores[sub_category_match] += self.sub_category_boost

        # ***
        for classification in classifications:
            tokenized_classification = snowball_tokenizer(classification)
            classification_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_classification:
                classification_match = self.index['classification_snowball'].array.score(tokenized_classification) > 0
            bm25_scores[classification_match] += self.direct_boost

        top_k = np.argsort(-bm25_scores)[:k]
        scores = bm25_scores[top_k]

        return top_k, scores

halluc_search = CategorySearch(corpus, classify)
graded_halluc = run_strategy(halluc_search, judgments)
graded_halluc

### Analyze results

* We notice a small bump in performance in NDCG from GPT-5.
* We also notice a bigger bump with Jev

In [ ]:
ndcgs(graded_bm25).mean(), ndcgs(graded_halluc).mean()